In [1]:
import duckdb
import pandas as pd
import os

In [6]:
data_precaire = pd.read_csv("../data/raw/pop_precaire.csv",sep=";",header=2)
data_precaire

,Code,Libellé,Part de la population en emploi précaire 2022
0,01001,L'Abergement-Clémenciat,9.9
1,01002,L'Abergement-de-Varey,8.3
2,01004,Ambérieu-en-Bugey,18
3,01005,Ambérieux-en-Dombes,10.6
4,01006,Ambléon,11.9
...,...,...,...
34872,97615,Pamandzi,N/A - résultat non disponible
34873,97616,Sada,N/A - résultat non disponible
34874,97617,Tsingoni,N/A - résultat non disponible
34875,97701,Saint-Barthélemy,23.9


In [8]:
df_precaire = data_precaire.rename(columns={"Code":"code_insee", "Libellé":"nom_com","Part de la population en emploi précaire 2022":"valeur" })
df_precaire

,code_insee,nom_com,valeur
0,01001,L'Abergement-Clémenciat,9.9
1,01002,L'Abergement-de-Varey,8.3
2,01004,Ambérieu-en-Bugey,18
3,01005,Ambérieux-en-Dombes,10.6
4,01006,Ambléon,11.9
...,...,...,...
34872,97615,Pamandzi,N/A - résultat non disponible
34873,97616,Sada,N/A - résultat non disponible
34874,97617,Tsingoni,N/A - résultat non disponible
34875,97701,Saint-Barthélemy,23.9


In [11]:
df_precaire[df_precaire["code_insee"].str.startswith('75')]

,code_insee,nom_com,valeur
29194,75056,Paris,16.8


In [10]:
df_epci = pd.read_csv("../data/processed/epci_membres.csv")
df_epci

,code_insee,nom,pop_tot_commune,pop_mun_commune,siren,epci_nom,epci_type,epci_modeFinancement,total_pop_tot,total_pop_mun,superficie_hectare,superficie_km2,dept_com,bassin_vie,dept_epci
0,01304,Pont-d'Ain,2912,2862,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1122.0,11.0,01,01304,01
1,01199,Jujurieux,2246,2209,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1548.0,15.0,01,01004,01
2,01363,Saint-Jean-le-Vieux,1873,1799,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1517.0,15.0,01,01004,01
3,01314,Priay,1826,1803,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1571.0,16.0,01,01004,01
4,01273,Neuville-sur-Ain,1823,1798,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1991.0,20.0,01,01004,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34996,97601,Acoua,5384,5192,200060465,CA du Grand Nord de Mayotte,CA,FPU,60372,59042,1290.0,13.0,97,97611,976
34997,97603,Bandrele,10529,10282,200060473,CC du Sud,CC,FPU,31945,30898,3514.0,35.0,97,97611,976
34998,97606,Chirongui,9197,8920,200060473,CC du Sud,CC,FPU,31945,30898,2590.0,26.0,97,97611,976
34999,97604,Bouéni,6503,6189,200060473,CC du Sud,CC,FPU,31945,30898,1381.0,14.0,97,97611,976


In [24]:
query = """ 
SELECT 
df_epci.dept_epci as dept_id,
df_epci.siren as epci_id,
df_epci.epci_nom AS epci_lib,
'i037' as id_indicator,
round(sum(df_epci.pop_mun_commune * TRY_CAST(df_precaire.valeur AS float)) / df_epci.total_pop_mun,2) as valeur_brute,
'2022' as annee
FROM df_epci
JOIN df_precaire
    ON df_epci.code_insee = df_precaire.code_insee
GROUP BY df_epci.dept_epci,df_epci.siren, df_epci.epci_nom,total_pop_mun
ORDER BY df_epci.dept_epci,df_epci.siren
"""

df_final = duckdb.query(query)

In [25]:
df_final

┌─────────┬───────────┬────────────────────────────────────────────────────────────┬──────────────┬──────────────┬─────────┐
│ dept_id │  epci_id  │                          epci_lib                          │ id_indicator │ valeur_brute │  annee  │
│ varchar │   int64   │                          varchar                           │   varchar    │    double    │ varchar │
├─────────┼───────────┼────────────────────────────────────────────────────────────┼──────────────┼──────────────┼─────────┤
│ 01      │ 200029999 │ CC Rives de l'Ain - Pays du Cerdon                         │ i037         │         14.8 │ 2022    │
│ 01      │ 200040350 │ CC Bugey Sud                                               │ i037         │        16.82 │ 2022    │
│ 01      │ 200042497 │ CC Dombes Saône Vallée                                     │ i037         │        10.67 │ 2022    │
│ 01      │ 200042935 │ CA Haut-Bugey Agglomération                                │ i037         │         19.2 │ 2022    │


In [27]:
df_final.write_csv("../data/processed/i164_precaire.csv")